In [1]:
# Практика 8. Групування і зведені таблиці
# Коць Артем, ІТ-42
# Варіант 9 — Суми

import numpy as np
import pandas as pd

np.random.seed(42)

base_temp = 7.5
amplitude = 14
city = "Суми"

rows = []

for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)

        rows.append({
            "місто": city,
            "рік": year,
            "місяць": month,
            "температура": round(base_temp + seasonal + noise, 1),
        })

climate = pd.DataFrame(rows)

climate

,місто,рік,місяць,температура
0,Суми,2021,1,-6.0
1,Суми,2021,2,-4.8
2,Суми,2021,3,1.1
3,Суми,2021,4,9.0
4,Суми,2021,5,14.3
5,Суми,2021,6,19.4
6,Суми,2021,7,23.1
7,Суми,2021,8,20.4
8,Суми,2021,9,14.0
9,Суми,2021,10,8.0


In [4]:
year_stats = climate.groupby("рік")["температура"].agg(
    ["mean", "min", "max"]
)

year_stats

,mean,min,max
рік,,,
2021,7.783333,-6.0,23.1
2022,6.916667,-6.5,20.6
2023,7.308333,-7.0,21.5
2024,7.158333,-6.6,21.4


### Завдання 1. Аналіз температури за роками

За результатами групування даних за роками було отримано середню,
мінімальну та максимальну температуру для кожного року.

У наборі даних за 2021–2024 роки немає чітко вираженого стабільного
тренду потепління або похолодання. Значення змінюються від року до
року, що пов'язано з випадковим шумом, доданим під час генерації
синтетичного набору.

Тому коливання температури між роками можна вважати переважно
випадковими, а не ознакою сталого кліматичного тренду.

In [5]:
month_stats = climate.groupby("місяць")["температура"].agg(
    ["mean", "std"]
)

month_stats

,mean,std
місяць,,
1,-6.400,0.424264
2,-5.600,1.104536
3,-0.400,1.023067
4,7.875,0.865544
5,14.225,0.727438
6,19.600,0.294392
7,21.500,1.116542
8,19.850,1.420094
9,14.375,1.250000


In [6]:
month_stats["std"].idxmax()

np.int64(8)

### Завдання 2. Аналіз температури за місяцями

Для кожного місяця було обчислено середню температуру та стандартне
відхилення за чотири роки.

Найбільше стандартне відхилення має місяць, який показав найбільший
розкид температур між 2021–2024 роками. Це можна пояснити випадковими
кліматичними коливаннями, які були додані під час генерації даних.

Перехідні сезони, зокрема весна та осінь, зазвичай є нестабільнішими,
оскільки температура в цей період може змінюватися сильніше. Тому
підвищений розкид у такому місяці є цілком логічним.

In [7]:
climate_pivot = climate.pivot_table(
    index="місяць",
    columns="рік",
    values="температура",
    aggfunc="mean"
)

climate_pivot

рік,2021,2022,2023,2024
місяць,,,,
1,-6.0,-6.3,-7.0,-6.3
2,-4.8,-6.5,-4.5,-6.6
3,1.1,-1.2,-0.7,-0.8
4,9.0,6.9,7.9,7.7
5,14.3,13.5,13.9,15.2
6,19.4,19.9,19.3,19.8
7,23.1,20.6,20.9,21.4
8,20.4,18.2,21.5,19.3
9,14.0,16.0,14.5,13.0


### Завдання 3. Аналіз pivot_table

Вихідний набір climate має довгий (tidy) формат: кожен рядок
відповідає одному спостереженню та містить місто, рік, місяць
і температуру.

За допомогою pivot_table() дані було перетворено у зведену таблицю,
де місяці розташовані в рядках, а роки — у стовпцях.

Для людини така зведена форма є зручнішою для швидкого порівняння
температури одного місяця між різними роками.

Довгий формат зручніший для подальшого групування, фільтрації,
агрегації та побудови графіків, оскільки всі спостереження мають
однакову структуру. Це відповідає принципам tidy data.

In [8]:
def get_season(month):
    if month in [12, 1, 2]:
        return "зима"
    elif month in [3, 4, 5]:
        return "весна"
    elif month in [6, 7, 8]:
        return "літо"
    else:
        return "осінь"


climate["сезон"] = climate["місяць"].apply(get_season)

climate["тепліше_за_середнє"] = (
    climate["температура"] > base_temp
)

climate.head(12)

,місто,рік,місяць,температура,сезон,тепліше_за_середнє
0,Суми,2021,1,-6.0,зима,False
1,Суми,2021,2,-4.8,зима,False
2,Суми,2021,3,1.1,весна,False
3,Суми,2021,4,9.0,весна,True
4,Суми,2021,5,14.3,весна,True
5,Суми,2021,6,19.4,літо,True
6,Суми,2021,7,23.1,літо,True
7,Суми,2021,8,20.4,літо,True
8,Суми,2021,9,14.0,осінь,True
9,Суми,2021,10,8.0,осінь,True


In [9]:
cross = pd.crosstab(
    climate["сезон"],
    climate["тепліше_за_середнє"]
)

cross

тепліше_за_середнє,False,True
сезон,,
весна,5,7
зима,12,0
літо,0,12
осінь,7,5


In [10]:
climate[climate["сезон"] == "літо"][
    ["місяць", "температура", "тепліше_за_середнє"]
]

,місяць,температура,тепліше_за_середнє
5,6,19.4,True
6,7,23.1,True
7,8,20.4,True
17,6,19.9,True
18,7,20.6,True
19,8,18.2,True
29,6,19.3,True
30,7,20.9,True
31,8,21.5,True
41,6,19.8,True


### Завдання 4. Аналіз crosstab

До набору даних було додано дві похідні категоріальні змінні:
"сезон" та "тепліше_за_середнє".

За допомогою crosstab() було підраховано кількість спостережень
для кожної комбінації сезону та категорії температури.

Розподіл відповідає очікуванням: улітку переважна більшість, а в
цьому синтетичному наборі всі температури мають бути вищими за
середньорічне значення 7.5 °C. Взимку, навпаки, переважають значення,
які не перевищують середньорічну температуру.

Це показує, що похідна категорія "тепліше_за_середнє" узгоджується
із сезонним характером температури.

In [11]:
climate_pivot_simple = climate.pivot(
    index="місяць",
    columns="рік",
    values="температура"
)

climate_pivot_simple

рік,2021,2022,2023,2024
місяць,,,,
1,-6.0,-6.3,-7.0,-6.3
2,-4.8,-6.5,-4.5,-6.6
3,1.1,-1.2,-0.7,-0.8
4,9.0,6.9,7.9,7.7
5,14.3,13.5,13.9,15.2
6,19.4,19.9,19.3,19.8
7,23.1,20.6,20.9,21.4
8,20.4,18.2,21.5,19.3
9,14.0,16.0,14.5,13.0


### Завдання 5. Аналіз pivot()

Метод pivot() успішно спрацював на наборі climate.

У наборі є 48 рядків: 4 роки × 12 місяців. Для кожної комбінації
"місяць + рік" існує рівно один рядок із температурою. Тому
дублікати комбінацій індексу та стовпця відсутні, і pivot() може
однозначно сформувати зведену таблицю.

У прикладі з кав'ярнями для однієї комбінації "філія + рік" було
чотири рядки, тому pivot() не міг визначити, яке значення використати
в одній клітинці.

Щоб pivot() почав завершуватися помилкою і для climate, потрібно
створити дублікати комбінацій "місяць + рік". Наприклад, можна додати
до міста кілька метеостанцій. Тоді для одного місяця та року буде
кілька температур, і pivot() видасть помилку через дублікати.

У такій ситуації доцільно використовувати pivot_table(), оскільки
вона дозволяє застосувати агрегувальну функцію, наприклад mean.

## Контрольні питання

### 1. Яка логіка split-apply-combine у groupby()?

`groupby()` працює за принципом split-apply-combine.

На першому етапі `split` дані розділяються на групи за значенням
певного стовпця, наприклад за роком.

На другому етапі `apply` до кожної групи застосовується потрібна
операція, наприклад обчислення середнього, мінімального або
максимального значення.

На третьому етапі `combine` результати всіх груп об'єднуються
в одну підсумкову таблицю.

### 2. У чому відмінність між pivot() і pivot_table()?

`pivot()` працює лише тоді, коли для кожної комбінації індексу
та стовпця існує одне унікальне значення.

Якщо для однієї комбінації існує декілька рядків, `pivot()` не може
визначити, яке значення потрібно помістити в клітинку, тому виникає
помилка через дублікати.

`pivot_table()` може працювати з дублікатиами, оскільки передбачає
агрегацію значень за допомогою `aggfunc`, наприклад `mean`, `sum`
або `count`.

### 3. Що показує crosstab() і чим він відрізняється від groupby().size()?

`crosstab()` показує частоти комбінацій категорій у вигляді
перехресної таблиці. Наприклад, у цій роботі він показує кількість
спостережень для кожного сезону та категорії
"тепліше_за_середнє".

`groupby(...).size()` також може підрахувати кількість рядків
у групах, але результат зазвичай має вигляд Series або таблиці
з ієрархічними групами.

`crosstab()` спеціально призначений для побудови таблиць частот
між категоріальними змінними.

### 4. Чому для groupby і pivot_table зручно використовувати довгий
(tidy) формат?

У довгому форматі кожен рядок є окремим спостереженням, а кожен
стовпець відповідає окремій змінній.

Такий формат зручний для `groupby()`, фільтрації, агрегування та
побудови графіків, оскільки всі спостереження мають однакову
структуру.

Якщо роки або місяці розміщені безпосередньо в назвах стовпців,
аналіз стає менш зручним. Тому для обробки даних краще зберігати
їх у tidy-форматі, а за необхідності створювати зведене
представлення за допомогою `pivot_table()`.